# Data prep de la partie comportement

Il ya eu des petits changement ( label et data aug avant le split )  pour cela on va repeter le travail
On continue avec le travail precedent pour ne pas perdre du temps a extraire les frames et importer data puisque j'ai enregistré ca sur drive

In [ ]:

import os
import shutil
import hashlib
import random
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ── Chemins ──────────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/projet_chiens'
FRAMES_DIR   = f'{DRIVE_BASE}/behavior/frames'   # frames existantes
OUTPUT_BASE  = f'{DRIVE_BASE}/behavior_v2'        # nouveau dossier
RESIZED_DIR  = f'{OUTPUT_BASE}/resized'
AUG_DIR      = f'{OUTPUT_BASE}/augmented'         # après augmentation
FINAL_DIR    = f'{OUTPUT_BASE}/final'             # après split
CSV_PATH     = f'{OUTPUT_BASE}/labels.csv'        # labels avec sous-classes

# ── Sous-classes par groupe ───────────────────────────────────
SOUS_CLASSES = {
    'normal'  : ['tail wagging', 'playing', 'sitting',
                 'standing', 'eating', 'lying'],
    'suspect' : ['Restlessness', 'Paralysis', 'Incoordination',
                 'digging', 'barking'],
    'anormal' : ['hyper salivation', 'bone in throat syndrome',
                 'dropped jaw', 'Sudden aggression', 'Seizure']
}

# ── Paramètres ────────────────────────────────────────────────
IMG_SIZE     = 224
SPLIT_RATIO  = (0.80, 0.10, 0.10)   # train / val / test
MIN_SIZE     = 50                    # pixels minimum
SEED         = 42

random.seed(SEED)

print("✅ Imports OK")
print(f"📁 Frames source : {FRAMES_DIR}")
print(f"📁 Output        : {OUTPUT_BASE}")

✅ Imports OK
📁 Frames source : /content/drive/MyDrive/projet_chiens/behavior/frames
📁 Output        : /content/drive/MyDrive/projet_chiens/behavior_v2


In [ ]:
# ============================================================
# CELLULE 2 — Montage Drive + Comptage des images
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

print("\n📊 Comptage des images par sous-classe :\n")

total = 0
for groupe, sous_classes in SOUS_CLASSES.items():
    print(f"  [{groupe.upper()}]")
    for sc in sous_classes:
        dossier = os.path.join(FRAMES_DIR, groupe, sc)
        if os.path.exists(dossier):
            imgs = [f for f in os.listdir(dossier)
                    if f.lower().endswith(('.jpg','.jpeg','.png'))]
            n = len(imgs)
            total += n
            print(f"    📁 {sc:40s} → {n} images")
        else:
            print(f"    ⚠️  {sc:40s} → DOSSIER INTROUVABLE")
    print()

print(f"  TOTAL : {total} images")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📊 Comptage des images par sous-classe :

  [NORMAL]
    📁 tail wagging                             → 118 images
    📁 playing                                  → 170 images
    ⚠️  sitting                                  → DOSSIER INTROUVABLE
    ⚠️  standing                                 → DOSSIER INTROUVABLE
    ⚠️  eating                                   → DOSSIER INTROUVABLE
    ⚠️  lying                                    → DOSSIER INTROUVABLE

  [SUSPECT]
    ⚠️  Restlessness                             → DOSSIER INTROUVABLE
    📁 Paralysis                                → 422 images
    📁 Incoordination                           → 272 images
    📁 digging                                  → 161 images
    📁 barking                                  → 164 images

  [ANORMAL]
    📁 hyper salivation                         → 150 images
    📁 bone in thr

In [ ]:
# ============================================================
# CELLULE 3 — Diagnostic : vrais noms des dossiers
# ============================================================
print("📂 Contenu réel de chaque groupe :\n")

for groupe in ['normal', 'suspect', 'anormal']:
    dossier_groupe = os.path.join(FRAMES_DIR, groupe)
    if os.path.exists(dossier_groupe):
        vrais_noms = os.listdir(dossier_groupe)
        print(f"  [{groupe.upper()}]")
        for nom in sorted(vrais_noms):
            print(f"    → '{nom}'")
    else:
        print(f"  ⚠️ Dossier {groupe} INTROUVABLE")
    print()

📂 Contenu réel de chaque groupe :

  [NORMAL]
    → 'playing'
    → 'roboflow_dogs_behavior_eating'
    → 'roboflow_dogs_behavior_lying'
    → 'roboflow_dogs_behavior_sitting'
    → 'roboflow_dogs_behavior_standing'
    → 'tail wagging'

  [SUSPECT]
    → 'Incoordination'
    → 'Paralysis'
    → 'Restlesness'
    → 'barking'
    → 'digging'
    → 'roboflow_dogs_behavior_barking'
    → 'roboflow_rabies_detection_Incoordination'
    → 'roboflow_rabies_detection_barking'
    → 'roboflow_rabies_detection_digging'

  [ANORMAL]
    → 'Sezure'
    → 'Sudden aggression'
    → 'bone in throat syndrome'
    → 'dropped jaw, toungh'
    → 'hyper salivation'
    → 'roboflow_rabies_detection_bone in throat syndrome'
    → 'roboflow_rabies_detection_dropped jaw_ toungh'
    → 'roboflow_rabies_detection_hyper_salivation'



In [ ]:
# ============================================================
# CELLULE 4 — Correction SOUS_CLASSES avec vrais noms
# ============================================================

# Chaque comportement peut venir de PLUSIEURS dossiers
# label_final → [dossier1, dossier2, ...]
SOUS_CLASSES_V2 = {
    'normal': {
        'tail_wagging' : ['tail wagging'],
        'playing'      : ['playing'],
        'sitting'      : ['roboflow_dogs_behavior_sitting'],
        'standing'     : ['roboflow_dogs_behavior_standing'],
        'eating'       : ['roboflow_dogs_behavior_eating'],
        'lying'        : ['roboflow_dogs_behavior_lying'],
    },
    'suspect': {
        'restlessness'    : ['Restlesness'],
        'paralysis'       : ['Paralysis'],
        'incoordination'  : ['Incoordination',
                             'roboflow_rabies_detection_Incoordination'],
        'digging'         : ['digging',
                             'roboflow_rabies_detection_digging'],
        'barking'         : ['barking',
                             'roboflow_dogs_behavior_barking',
                             'roboflow_rabies_detection_barking'],
    },
    'anormal': {
        'hyper_salivation'        : ['hyper salivation',
                                     'roboflow_rabies_detection_hyper_salivation'],
        'bone_in_throat'          : ['bone in throat syndrome',
                                     'roboflow_rabies_detection_bone in throat syndrome'],
        'dropped_jaw'             : ['dropped jaw, toungh',
                                     'roboflow_rabies_detection_dropped jaw_ toungh'],
        'sudden_aggression'       : ['Sudden aggression'],
        'seizure'                 : ['Sezure'],
    }
}

# ── Vérification comptage ─────────────────────────────────────
print("📊 Comptage après regroupement :\n")
total = 0
for groupe, comportements in SOUS_CLASSES_V2.items():
    print(f"  [{groupe.upper()}]")
    for label, dossiers in comportements.items():
        n = 0
        for d in dossiers:
            chemin = os.path.join(FRAMES_DIR, groupe, d)
            if os.path.exists(chemin):
                imgs = [f for f in os.listdir(chemin)
                        if f.lower().endswith(('.jpg','.jpeg','.png'))]
                n += len(imgs)
            else:
                print(f"    ⚠️  INTROUVABLE : {groupe}/{d}")
        print(f"    📁 {label:30s} → {n} images")
        total += n
    print()
print(f"  TOTAL : {total} images")

📊 Comptage après regroupement :

  [NORMAL]
    📁 tail_wagging                   → 118 images
    📁 playing                        → 170 images
    📁 sitting                        → 76 images
    📁 standing                       → 90 images
    📁 eating                         → 52 images
    📁 lying                          → 97 images

  [SUSPECT]
    📁 restlessness                   → 206 images
    📁 paralysis                      → 422 images
    📁 incoordination                 → 323 images
    📁 digging                        → 1006 images
    📁 barking                        → 1986 images

  [ANORMAL]
    📁 hyper_salivation               → 282 images
    📁 bone_in_throat                 → 270 images
    📁 dropped_jaw                    → 468 images
    📁 sudden_aggression              → 410 images
    📁 seizure                        → 223 images

  TOTAL : 6199 images


## Définir la fonction d'augmentation
 seed pour une transformation unique de chaque image , changer les parametres pour ne pas avoir de doublons apres


In [ ]:
# ============================================================
# CELLULE 5A — Fonction augmentation (combinaisons uniques)
# ============================================================
from PIL import Image, ImageEnhance, ImageFilter
from itertools import combinations

def augment_image_unique(img, combo_id):
    """
    combo_id unique → combinaison unique → jamais de doublon
    Pas besoin de seed !
    """
    transformations = [
        lambda x: x.transpose(Image.FLIP_LEFT_RIGHT),
        lambda x: x.transpose(Image.FLIP_TOP_BOTTOM),
        lambda x: x.rotate(15),
        lambda x: x.rotate(-15),
        lambda x: x.rotate(30),
        lambda x: x.rotate(-30),
        lambda x: ImageEnhance.Brightness(x).enhance(1.3),
        lambda x: ImageEnhance.Brightness(x).enhance(0.7),
        lambda x: ImageEnhance.Contrast(x).enhance(1.3),
        lambda x: ImageEnhance.Contrast(x).enhance(0.7),
        lambda x: x.filter(ImageFilter.GaussianBlur(1)),
        lambda x: ImageEnhance.Sharpness(x).enhance(2.0),
    ]

    # 12 transformations → 66 paires uniques possibles
    combos = list(combinations(range(len(transformations)), 2))

    combo = combos[combo_id % len(combos)]
    for i in combo:
        img = transformations[i](img)

    return img

# Vérification
from itertools import combinations
n_combos = len(list(combinations(range(12), 2)))
print(f"✅ Fonction augmentation définie")
print(f"ℹ️  {n_combos} combinaisons uniques possibles → pas de doublons")

✅ Fonction augmentation définie
ℹ️  66 combinaisons uniques possibles → pas de doublons


## Collecter toutes les images par comportement et definir target 400 apres avoir fait la moyenne du toutes ces dernieres.

In [ ]:


# Dictionnaire final : label → liste de chemins d'images
TARGET = 400
images_par_label = {}

print("📂 Collecte des images par comportement :\n")

for groupe, comportements in SOUS_CLASSES_V2.items():
    for label, dossiers in comportements.items():

        chemins = []
        for d in dossiers:
            chemin = os.path.join(FRAMES_DIR, groupe, d)
            if os.path.exists(chemin):
                for fname in os.listdir(chemin):
                    if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                        chemins.append(os.path.join(chemin, fname))

        images_par_label[label] = {
            'groupe'  : groupe,   # normal / suspect / anormal
            'chemins' : chemins,  # liste des chemins
            'count'   : len(chemins)
        }

        status = "🔧 augmenter" if len(chemins) < TARGET else "✅ ok"
        print(f"  {status} | {groupe:8s} | {label:25s} → {len(chemins)} images")

print(f"\n✅ Collecte terminée — {len(images_par_label)} comportements trouvés")

📂 Collecte des images par comportement :

  🔧 augmenter | normal   | tail_wagging              → 118 images
  🔧 augmenter | normal   | playing                   → 170 images
  🔧 augmenter | normal   | sitting                   → 76 images
  🔧 augmenter | normal   | standing                  → 90 images
  🔧 augmenter | normal   | eating                    → 52 images
  🔧 augmenter | normal   | lying                     → 97 images
  🔧 augmenter | suspect  | restlessness              → 206 images
  ✅ ok | suspect  | paralysis                 → 422 images
  🔧 augmenter | suspect  | incoordination            → 323 images
  ✅ ok | suspect  | digging                   → 1006 images
  ✅ ok | suspect  | barking                   → 1986 images
  🔧 augmenter | anormal  | hyper_salivation          → 282 images
  🔧 augmenter | anormal  | bone_in_throat            → 270 images
  ✅ ok | anormal  | dropped_jaw               → 468 images
  ✅ ok | anormal  | sudden_aggression         → 410 images
  🔧 a

In [ ]:
# ============================================================
# CELLULE 6B — Vérification taille + corrompues sur originaux
# ============================================================
from PIL import Image

print("🔍 Vérification des images originales...\n")

total_supprimees = 0

for groupe, comportements in SOUS_CLASSES_V2.items():
    for label, dossiers in comportements.items():
        supprimees = 0
        total = 0

        for d in dossiers:
            chemin = os.path.join(FRAMES_DIR, groupe, d)
            if not os.path.exists(chemin):
                continue

            for fname in os.listdir(chemin):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                total += 1
                fpath = os.path.join(chemin, fname)
                try:
                    img = Image.open(fpath)
                    w, h = img.size
                    if w < MIN_SIZE or h < MIN_SIZE:
                        os.remove(fpath)
                        supprimees += 1
                except:
                    os.remove(fpath)
                    supprimees += 1

        total_supprimees += supprimees
        status = "⚠️ " if supprimees > 0 else "✅"
        print(f"  {status} {label:25s} → {total} images | {supprimees} supprimées")

print(f"\n  Total supprimées : {total_supprimees}")
print("✅ Vérification terminée !")

🔍 Vérification des images originales...

  ✅ tail_wagging              → 118 images | 0 supprimées
  ✅ playing                   → 170 images | 0 supprimées
  ✅ sitting                   → 80 images | 0 supprimées
  ✅ standing                  → 95 images | 0 supprimées
  ✅ eating                    → 53 images | 0 supprimées
  ✅ lying                     → 101 images | 0 supprimées
  ✅ restlessness              → 206 images | 0 supprimées
  ✅ paralysis                 → 422 images | 0 supprimées
  ✅ incoordination            → 330 images | 0 supprimées
  ✅ digging                   → 1006 images | 0 supprimées
  ✅ barking                   → 1987 images | 0 supprimées
  ✅ hyper_salivation          → 282 images | 0 supprimées
  ✅ bone_in_throat            → 270 images | 0 supprimées
  ✅ dropped_jaw               → 468 images | 0 supprimées
  ✅ sudden_aggression         → 410 images | 0 supprimées
  ✅ seizure                   → 223 images | 0 supprimées

  Total supprimées : 0
✅ Vérifi

In [ ]:
# ============================================================
# CELLULE 6C — Déduplication MD5 sur originaux
# ============================================================
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("🔍 Déduplication MD5 en cours...\n")

total_doublons = 0

for groupe, comportements in SOUS_CLASSES_V2.items():
    for label, dossiers in comportements.items():

        hashes_vus = {}  # hash → premier fichier vu
        doublons   = 0

        for d in dossiers:
            chemin = os.path.join(FRAMES_DIR, groupe, d)
            if not os.path.exists(chemin):
                continue

            for fname in os.listdir(chemin):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                fpath = os.path.join(chemin, fname)
                h     = md5_hash(fpath)

                if h in hashes_vus:
                    os.remove(fpath)  # doublon → supprimer
                    doublons += 1
                else:
                    hashes_vus[h] = fpath

        total_doublons += doublons
        status = "⚠️ " if doublons > 0 else "✅"
        print(f"  {status} {label:25s} → {doublons} doublons supprimés")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Déduplication terminée !")

🔍 Déduplication MD5 en cours...

  ✅ tail_wagging              → 0 doublons supprimés
  ✅ playing                   → 0 doublons supprimés
  ⚠️  sitting                   → 4 doublons supprimés
  ⚠️  standing                  → 5 doublons supprimés
  ⚠️  eating                    → 1 doublons supprimés
  ⚠️  lying                     → 4 doublons supprimés
  ✅ restlessness              → 0 doublons supprimés
  ✅ paralysis                 → 0 doublons supprimés
  ⚠️  incoordination            → 7 doublons supprimés
  ✅ digging                   → 0 doublons supprimés
  ⚠️  barking                   → 1 doublons supprimés
  ✅ hyper_salivation          → 0 doublons supprimés
  ✅ bone_in_throat            → 0 doublons supprimés
  ✅ dropped_jaw               → 0 doublons supprimés
  ✅ sudden_aggression         → 0 doublons supprimés
  ✅ seizure                   → 0 doublons supprimés

  Total doublons : 22
✅ Déduplication terminée !


In [ ]:
# ============================================================
# CELLULE 6D — Resize 224×224 sur les originaux
# ============================================================
from PIL import Image
import os

# Dossier pour stocker les images redimensionnées
RESIZED_DIR = f'{OUTPUT_BASE}/resized'
os.makedirs(RESIZED_DIR, exist_ok=True)

print("📐 Resize 224×224 en cours...\n")

for groupe, comportements in SOUS_CLASSES_V2.items():
    for label, dossiers in comportements.items():

        dest_dir = os.path.join(RESIZED_DIR, label)
        os.makedirs(dest_dir, exist_ok=True)

        n_ok    = 0
        n_erreur = 0

        for d in dossiers:
            chemin = os.path.join(FRAMES_DIR, groupe, d)
            if not os.path.exists(chemin):
                continue

            for fname in os.listdir(chemin):
                if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                fpath = os.path.join(chemin, fname)
                try:
                    img = Image.open(fpath).convert('RGB')
                    img = img.resize((IMG_SIZE, IMG_SIZE))
                    dst = os.path.join(dest_dir, fname)
                    img.save(dst, 'JPEG', quality=95)
                    n_ok += 1
                except:
                    n_erreur += 1

        print(f"  ✅ {label:25s} → {n_ok} images redimensionnées "
              f"| {n_erreur} erreurs")

print(f"\n✅ Resize terminé !")
print(f"📁 Résultat dans : {RESIZED_DIR}")

📐 Resize 224×224 en cours...

  ✅ tail_wagging              → 118 images redimensionnées | 0 erreurs
  ✅ playing                   → 170 images redimensionnées | 0 erreurs
  ✅ sitting                   → 76 images redimensionnées | 0 erreurs
  ✅ standing                  → 90 images redimensionnées | 0 erreurs
  ✅ eating                    → 52 images redimensionnées | 0 erreurs
  ✅ lying                     → 97 images redimensionnées | 0 erreurs
  ✅ restlessness              → 206 images redimensionnées | 0 erreurs
  ✅ paralysis                 → 422 images redimensionnées | 0 erreurs
  ✅ incoordination            → 323 images redimensionnées | 0 erreurs
  ✅ digging                   → 1006 images redimensionnées | 0 erreurs
  ✅ barking                   → 1986 images redimensionnées | 0 erreurs
  ✅ hyper_salivation          → 282 images redimensionnées | 0 erreurs
  ✅ bone_in_throat            → 270 images redimensionnées | 0 erreurs
  ✅ dropped_jaw               → 468 images redime

## Augmentation (oversampling)
on copie les images originaux dans AUG_DIR pour faire l'augmentation sur eux  et pour ne perdre (originaux)

In [ ]:
import shutil
if os.path.exists(AUG_DIR):
    shutil.rmtree(AUG_DIR)
    print("✅ Ancien AUG_DIR supprimé")

✅ Ancien AUG_DIR supprimé


In [ ]:
# ============================================================
# CELLULE 6E — Augmentation (combinaisons uniques)
# ============================================================
import shutil

os.makedirs(AUG_DIR, exist_ok=True)

print(f"🎯 Cible : {TARGET} images par comportement\n")
print("🔄 Augmentation en cours...\n")

for label in images_par_label.keys():

    src_dir  = os.path.join(RESIZED_DIR, label)
    dest_dir = os.path.join(AUG_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    all_imgs = sorted([f for f in os.listdir(src_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    n_orig = len(all_imgs)

    # ── Étape 1 : Copier les originaux ──────────────────────
    for fname in all_imgs:
        shutil.copy2(os.path.join(src_dir, fname),
                     os.path.join(dest_dir, fname))

    # ── Étape 2 : Augmenter si nécessaire ───────────────────
    n_aug    = 0
    combo_id = 0
    if n_orig < TARGET:
        needed = TARGET - n_orig
        idx    = 0
        while n_aug < needed:
            fname = all_imgs[idx % len(all_imgs)]
            fpath = os.path.join(src_dir, fname)
            try:
                img = Image.open(fpath).convert('RGB')
                img = augment_image_unique(img, combo_id)
                img = img.resize((IMG_SIZE, IMG_SIZE))
                dst = os.path.join(dest_dir,
                                   f'{label}_aug{n_aug:04d}.jpg')
                img.save(dst, 'JPEG', quality=95)
                n_aug    += 1
                combo_id += 1  # combo différent à chaque fois ✅
            except:
                pass
            idx += 1

    total  = n_orig + n_aug
    status = "✅" if n_aug == 0 else "🔧"
    print(f"  {status} {label:25s} {n_orig:4d} → {total:4d} (+{n_aug} aug)")

print(f"\n✅ Augmentation terminée !")
print(f"📁 Résultat dans : {AUG_DIR}")

🎯 Cible : 400 images par comportement

🔄 Augmentation en cours...

  🔧 tail_wagging               118 →  400 (+282 aug)
  🔧 playing                    170 →  400 (+230 aug)
  🔧 sitting                     76 →  400 (+324 aug)
  🔧 standing                    90 →  400 (+310 aug)
  🔧 eating                      52 →  400 (+348 aug)
  🔧 lying                       97 →  400 (+303 aug)
  🔧 restlessness               206 →  400 (+194 aug)
  ✅ paralysis                  422 →  422 (+0 aug)
  🔧 incoordination             323 →  400 (+77 aug)
  ✅ digging                   1006 → 1006 (+0 aug)
  ✅ barking                   1986 → 1986 (+0 aug)
  🔧 hyper_salivation           282 →  400 (+118 aug)
  🔧 bone_in_throat             270 →  400 (+130 aug)
  ✅ dropped_jaw                468 →  468 (+0 aug)
  ✅ sudden_aggression          410 →  410 (+0 aug)
  🔧 seizure                    223 →  400 (+177 aug)

✅ Augmentation terminée !
📁 Résultat dans : /content/drive/MyDrive/projet_chiens/behavior_v2/au

In [ ]:
# ============================================================
# CELLULE 6F — Undersampling à 400 (après augmentation)
# ============================================================
import random

random.seed(SEED)

print(f"✂️  Undersampling à {TARGET} en cours...\n")

for label in images_par_label.keys():

    src_dir  = os.path.join(AUG_DIR, label)
    all_imgs = [f for f in os.listdir(src_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    n_avant = len(all_imgs)

    if n_avant > TARGET:
        # Garder en priorité les originaux (pas les augmentées)
        originaux  = [f for f in all_imgs if '_aug' not in f]
        augmentees = [f for f in all_imgs if '_aug' in f]

        # Prendre tous les originaux si possible
        if len(originaux) >= TARGET:
            garder = random.sample(originaux, TARGET)
        else:
            # Compléter avec des augmentées
            reste  = TARGET - len(originaux)
            garder = originaux + random.sample(augmentees, reste)

        # Supprimer les fichiers non gardés
        garder_set = set(garder)
        for fname in all_imgs:
            if fname not in garder_set:
                os.remove(os.path.join(src_dir, fname))

    n_apres = len([f for f in os.listdir(src_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    status = "✂️ " if n_avant > TARGET else "✅"
    print(f"  {status} {label:25s} {n_avant:4d} → {n_apres:4d}")

print(f"\n✅ Undersampling terminé !")
print(f"📊 Toutes les classes : {TARGET} images maximum")

✂️  Undersampling à 400 en cours...

  ✅ tail_wagging               400 →  400
  ✅ playing                    400 →  400
  ✅ sitting                    400 →  400
  ✅ standing                   400 →  400
  ✅ eating                     400 →  400
  ✅ lying                      400 →  400
  ✅ restlessness               400 →  400
  ✂️  paralysis                  422 →  400
  ✅ incoordination             400 →  400
  ✂️  digging                   1006 →  400
  ✂️  barking                   1986 →  400
  ✅ hyper_salivation           400 →  400
  ✅ bone_in_throat             400 →  400
  ✂️  dropped_jaw                468 →  400
  ✂️  sudden_aggression          410 →  400
  ✅ seizure                    400 →  400

✅ Undersampling terminé !
📊 Toutes les classes : 400 images maximum


In [ ]:
# ============================================================
# VÉRIFICATION — Doublons après augmentation + undersampling
# ============================================================
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("🔍 Vérification doublons dans AUG_DIR...\n")

total_doublons = 0
labels = sorted(os.listdir(AUG_DIR))

for label in labels:
    src_dir  = os.path.join(AUG_DIR, label)
    all_imgs = [f for f in os.listdir(src_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    hashes_vus = {}
    doublons   = 0

    for fname in all_imgs:
        fpath = os.path.join(src_dir, fname)
        h     = md5_hash(fpath)

        if h in hashes_vus:
            doublons += 1
        else:
            hashes_vus[h] = fname

    total_doublons += doublons
    status = "⚠️ " if doublons > 0 else "✅"
    print(f"  {status} {label:25s} → {doublons} doublons")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Vérification terminée !")

🔍 Vérification doublons dans AUG_DIR...

  ✅ barking                   → 0 doublons
  ✅ bone_in_throat            → 0 doublons
  ✅ digging                   → 0 doublons
  ✅ dropped_jaw               → 0 doublons
  ✅ eating                    → 0 doublons
  ✅ hyper_salivation          → 0 doublons
  ✅ incoordination            → 0 doublons
  ✅ lying                     → 0 doublons
  ✅ paralysis                 → 0 doublons
  ✅ playing                   → 0 doublons
  ✅ restlessness              → 0 doublons
  ✅ seizure                   → 0 doublons
  ✅ sitting                   → 0 doublons
  ✅ standing                  → 0 doublons
  ✅ sudden_aggression         → 0 doublons
  ✅ tail_wagging              → 0 doublons

  Total doublons : 0
✅ Vérification terminée !


In [ ]:
# ============================================================
# Nettoyage — Supprimer ancien final/ sur Drive
# ============================================================
import shutil

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)
    print(f"✅ Supprimé : {FINAL_DIR}")
else:
    print(f"ℹ️  Déjà absent : {FINAL_DIR}")

✅ Supprimé : /content/drive/MyDrive/projet_chiens/behavior_v2/final


In [ ]:
# ============================================================
# CELLULE 7 — Split 80/10/10
# ============================================================
import random

random.seed(SEED)

# Créer dossiers train/val/test
for split in ['train', 'val', 'test']:
    for label in images_par_label.keys():
        os.makedirs(os.path.join(FINAL_DIR, split, label), exist_ok=True)

print("📊 Split 80/10/10 en cours...\n")

for label in images_par_label.keys():

    src_dir  = os.path.join(AUG_DIR, label)
    all_imgs = [f for f in os.listdir(src_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    random.shuffle(all_imgs)

    n       = len(all_imgs)
    n_train = int(n * 0.80)  # 320 images
    n_val   = int(n * 0.10)  # 40 images

    splits = {
        'train' : all_imgs[:n_train],
        'val'   : all_imgs[n_train:n_train + n_val],
        'test'  : all_imgs[n_train + n_val:]
    }

    for split, files in splits.items():
        for fname in files:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(FINAL_DIR, split, label, fname)
            shutil.copy2(src, dst)

    print(f"  ✅ {label:25s} → "
          f"train:{len(splits['train']):3d} | "
          f"val:{len(splits['val']):3d} | "
          f"test:{len(splits['test']):3d}")

print(f"\n✅ Split terminé !")
print(f"📁 Résultat dans : {FINAL_DIR}")

📊 Split 80/10/10 en cours...

  ✅ tail_wagging              → train:320 | val: 40 | test: 40
  ✅ playing                   → train:320 | val: 40 | test: 40
  ✅ sitting                   → train:320 | val: 40 | test: 40
  ✅ standing                  → train:320 | val: 40 | test: 40
  ✅ eating                    → train:320 | val: 40 | test: 40
  ✅ lying                     → train:320 | val: 40 | test: 40
  ✅ restlessness              → train:320 | val: 40 | test: 40
  ✅ paralysis                 → train:320 | val: 40 | test: 40
  ✅ incoordination            → train:320 | val: 40 | test: 40
  ✅ digging                   → train:320 | val: 40 | test: 40
  ✅ barking                   → train:320 | val: 40 | test: 40
  ✅ hyper_salivation          → train:320 | val: 40 | test: 40
  ✅ bone_in_throat            → train:320 | val: 40 | test: 40
  ✅ dropped_jaw               → train:320 | val: 40 | test: 40
  ✅ sudden_aggression         → train:320 | val: 40 | test: 40
  ✅ seizure              

In [ ]:
# Supprimer ancien labels.csv
import os

if os.path.exists(CSV_PATH):
    os.remove(CSV_PATH)
    print(f"✅ Supprimé : {CSV_PATH}")
else:
    print(f"ℹ️  Déjà absent : {CSV_PATH}")

✅ Supprimé : /content/drive/MyDrive/projet_chiens/behavior_v2/labels.csv


In [ ]:
# ============================================================
# CELLULE 8 — Génération labels.csv (normal / anormal)
# ============================================================
import pandas as pd

print("📝 Génération labels.csv...\n")

# Mapping label → groupe (normal / anormal)
label_to_groupe = {
    # NORMAL
    'tail_wagging'      : 'normal',
    'playing'           : 'normal',
    'sitting'           : 'normal',
    'standing'          : 'normal',
    'eating'            : 'normal',
    'lying'             : 'normal',
    # ANORMAL (suspect + anormal fusionnés)
    'restlessness'      : 'anormal',
    'paralysis'         : 'anormal',
    'incoordination'    : 'anormal',
    'digging'           : 'anormal',
    'barking'           : 'anormal',
    'hyper_salivation'  : 'anormal',
    'bone_in_throat'    : 'anormal',
    'dropped_jaw'       : 'anormal',
    'sudden_aggression' : 'anormal',
    'seizure'           : 'anormal',
}

records = []

for split in ['train', 'val', 'test']:
    for label in images_par_label.keys():
        dossier = os.path.join(FINAL_DIR, split, label)
        for fname in os.listdir(dossier):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({
                    'filepath' : os.path.join(split, label, fname),
                    'label'    : label,                    # ex: barking
                    'groupe'   : label_to_groupe[label],   # normal / anormal
                    'split'    : split
                })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print(f"✅ labels.csv → {len(df)} lignes\n")
print("📊 Distribution par groupe :")
print(df.groupby(['split', 'groupe']).size().unstack(fill_value=0))
print("\n📊 Distribution par label :")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
print(f"\n📁 Sauvegardé dans : {CSV_PATH}")

📝 Génération labels.csv...

✅ labels.csv → 6400 lignes

📊 Distribution par groupe :
groupe  anormal  normal
split                  
test        400     240
train      3200    1920
val         400     240

📊 Distribution par label :
label  barking  bone_in_throat  digging  dropped_jaw  eating  \
split                                                          
test        40              40       40           40      40   
train      320             320      320          320     320   
val         40              40       40           40      40   

label  hyper_salivation  incoordination  lying  paralysis  playing  \
split                                                                
test                 40              40     40         40       40   
train               320             320    320        320      320   
val                  40              40     40         40       40   

label  restlessness  seizure  sitting  standing  sudden_aggression  \
split                     

In [ ]:
# Supprimer ancien ZIP
old_zip = f'{OUTPUT_BASE}/behavior_v2_final.zip'
if os.path.exists(old_zip):
    os.remove(old_zip)
    print(f"✅ Ancien ZIP supprimé")
else:
    print(f"ℹ️  Déjà absent")

✅ Ancien ZIP supprimé


In [ ]:
# ============================================================
# CELLULE 9 — ZIP final
# ============================================================
import shutil

ZIP_PATH = f'{OUTPUT_BASE}/behavior_v2_final'

print(" Compression en cours...")

shutil.make_archive(
    ZIP_PATH,   # nom du zip
    'zip',      # format
    OUTPUT_BASE, # dossier parent
    'final'     # dossier à zipper
)

# Vérifier la taille
size = os.path.getsize(ZIP_PATH + '.zip')
print(f"✅ ZIP créé : {ZIP_PATH}.zip")
print(f"📁 Taille   : {size / (1024*1024):.1f} MB")

 Compression en cours...
✅ ZIP créé : /content/drive/MyDrive/projet_chiens/behavior_v2/behavior_v2_final.zip
📁 Taille   : 119.1 MB
